# Step 5 · Train the world model on Kaggle (configs/world_model_v1.yaml)

Settings: **Accelerator → GPU T4 x2**, **Internet → On**, **Add Input → Datasets → Your Datasets →** the main dataset **and** the DAgger dataset.

W&B: same secret as Step 4 (Add-ons → Secrets, ticked for this notebook).

**Continuing an earlier run:** Add Input → Notebook Output → this notebook's previous version. Cell 4 finds its `last.pt` and resumes.

Afterwards download `world_model.zip` from the Output panel.

In [ ]:
# 1. Fresh clone of the repo
%cd /kaggle/working
!rm -rf robot-world-model
!git clone -q https://github.com/Swayam-27/robot-world-model.git
%cd /kaggle/working/robot-world-model
!git log --oneline -1

In [ ]:
# 2. W&B login from the Kaggle secret (use the exact secret name you gave it in Step 4)
!pip install -q wandb
import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("W&B key loaded")

In [ ]:
# 3. Find both datasets (folders that hold episodes.csv); the DAgger one has 'dagger' in its path
import glob
hits = [os.path.dirname(h) for h in glob.glob("/kaggle/input/**/episodes.csv", recursive=True)]
main = [h for h in hits if "dagger" not in h.lower()]
dagger = [h for h in hits if "dagger" in h.lower()]
assert len(main) == 1 and len(dagger) == 1, f"attach both datasets: main {main}, dagger {dagger}"
DATA, DAGGER = main[0], dagger[0]
print("main:", DATA, len(glob.glob(DATA + "/ep_*.npz")), "episodes | dagger:", DAGGER, len(glob.glob(DAGGER + "/ep_*.npz")), "episodes")

In [ ]:
# 4. Train. Stops after max_hours in the config (first run: 3 h) or max_steps; saves last.pt every 30 min.
#    If a previous version's output is attached, resume from its last.pt.
ckpts = glob.glob("/kaggle/input/**/world_model_v1/last.pt", recursive=True)
RESUME = f'--resume "{ckpts[0]}"' if ckpts else ""
print("resuming from", ckpts[0] if ckpts else "nothing: fresh start")
!python -m models.train_world_model --config configs/world_model_v1.yaml --data-dir "$DATA" --extra-data-dir "$DAGGER" $RESUME

In [ ]:
# 5. Pack the results (download world_model.zip from the Output panel): checkpoints, eval log, all sample grids
!cd outputs && zip -qr /kaggle/working/world_model.zip world_model_v1 && unzip -l /kaggle/working/world_model.zip | tail -3